# AI-Powered Ticket Processing Pipeline - Phase1: Prompt Testing
**Objective:** Evaluate LLM prompt strategies using Chain of Thought (CoT) and enforce structured JSON outputs using Pydantic on the 50-sample dataset.

## 1. Environment Setup and Configuration
Importing required dependencies and configuring the LLM client and projects paths.

In [36]:
import os
import json
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

In [37]:
try:
  from google.colab import userdata # pyright: ignore[reportMissingImports] 
  API_KEY = userdata.get("OPENAI_API_KEY")

except:
  load_dotenv()
  API_KEY = os.getenv("OPENAI_API_KEY")

CONFIG = {
    "input_sample_file" : "data/poc_sample_50.csv",
    "output_results_file" : "data/poc_results_50.csv",
    "text_col" : "response",
    "model_name" : "gpt-5.6-sol",
    "base_url" : "https://api.gapgpt.app/v1",
    "api_key" : API_KEY
}

In [38]:
#  Initialize LLM client
client = OpenAI(
    api_key = CONFIG["api_key"],
    base_url = CONFIG["base_url"]
)

## 2. Load Sampled Dataset
Loading the strategic 50-sample dataset generated in the previous PoC phase.

In [39]:
df_sample = pd.read_csv(CONFIG["input_sample_file"])

print(f"Loaded dataset shape:{df_sample.shape}")
df_sample.head()

Loaded dataset shape:(50, 7)


,flags,instruction,category,intent,response,text_length,noise_score
0,BCL,"I want to check your money back guarantee, I n...",REFUND,check_refund_policy,Definitely! I completely understand your desir...,2424,0.035066
1,BL,help me seeing what hours I can contact custom...,CONTACT,contact_customer_service,Grateful for your contact! I get the sense tha...,457,0.024070
2,BCIL,"I have got to use the standard profile, can I ...",ACCOUNT,switch_account,For sure! I'm here to provide you with the sup...,831,0.108303
3,BL,I want assistance making a complaint against y...,FEEDBACK,complaint,I'm sorry to hear that you've had a negative e...,630,0.019048
4,BILQ,can ya help me to check ur reimbursement policy,REFUND,check_refund_policy,For sure! I understand your request to check o...,2366,0.032544


## 3. Define Output Schema (Pydantic)
Defin the expected JSON structure for the LLM output, including Moderation, Classification, Sentiment, and Summarization.

In [40]:
class TicketProcessingResult(BaseModel):
    is_safe: bool = Field(description="True if the ticket contains no harmful, offensive, or unsafe content (Moderation).")
    category: str = Field(description="The main category of the ticket (e.g., Billing, Technical Support, Account, General Inquiry).")
    sentiment: str = Field(description="The emotional tone of the ticket: Positive, Neutral, or Negative.")
    summary: str = Field(description="A concise, one-sentence summary of the user's issue.")
    suggested_action: str = Field(description="A brief recommended next step for the support agent.")
    reasoning: str = Field(description="Chain of Thought: Step-by-step reasoning explaining how the category, sentiment, and safety were determined.")

## 4. Prompt Engineering (System & User Prompts)
Designing the Chain of Though (CoT) system prompt to instruct the model.

In [41]:
SYSTEM_PROMPT = """You are an expert AI customer support assistant.
Your task is to analyze customer support tickets and extract key information.

Analyze the ticket step-by-step (Chain of Though):
1. First, check if the content is safe and appropriate (Moderation).
2. Determine the core topic to assign a category (Classification).
3. Evaluate the tone of the text (Sentiment Analysis).
4. Summarize the issue in one sentence (Summarization).
5. Suggest a logical next step to resolve the issue.

You MUST return your response entirely in valid JSON format matching the exact keys provided by the user. Do not include Markdown blocks like   json.
"""

In [42]:
def create_user_prompt(ticket_text:str) -> str:
  schema = TicketProcessingResult.model_json_schema()
  return f"Please analyze the following ticket and return a JSON object matching this schema:\n\n{json.dumps(schema, indent=2)}\n\nTicket Text:\n\"\"\"{ticket_text}\"\"\""


## 5. Single Output Testing
Validating the prompt logic and JSON parsing on a single ticket before batch processing.


In [43]:
# Select a sample ticket
sample_text = df_sample.iloc[0][CONFIG["text_col"]]
print(f"Sample Ticket:\n{sample_text}")

Sample Ticket:
Definitely! I completely understand your desire to verify our money back guarantee and ensure that you're well-informed. Our money back guarantee is designed to provide you with peace of mind and protect your interests. Here's a breakdown of how it works:

1. **Product/Service Dissatisfaction:** If you're unsatisfied with the product or service you purchased, we offer a money back guarantee within a specified timeframe. Simply reach out to our customer support team, provide details about your dissatisfaction, and they'll guide you through the refund process.

2. **Quality or Performance Issues:** If you encounter any quality issues or the product or service you receive doesn't perform as expected, rest assured that you may be eligible for a refund. We take pride in delivering reliable and high-quality products/services, and we stand behind our commitment to your satisfaction.

3. **Delivery Delays or Non-Delivery:** We understand that timely delivery is crucial, and we s

In [44]:
# Call the model
response = client.chat.completions.create(
    model = CONFIG["model_name"],
    messages = [
        {"role" : "system", "content": SYSTEM_PROMPT},
        {"role" : "user", "content" : create_user_prompt(sample_text)}
    ],
    temperature = 0.0,
    response_format = {"type" : "json_object"}
)

In [45]:
# Parse and validate using Pydantic
raw_output = response.choices[0].message.content

try:
    parsed_result = TicketProcessingResult.model_validate_json(raw_output)
    print("Successfully Parsed Output:\n")
    print(parsed_result.model_dump_json(indent=2))
except Exception as e:
    print(f"Parsing Error: {e}")
    print(f"Raw Output was:\n{raw_output}")

Successfully Parsed Output:

{
  "is_safe": true,
  "category": "Billing",
  "sentiment": "Positive",
  "summary": "The ticket provides general information about the money-back guarantee and invites the customer to submit purchase details for personalized refund assistance.",
  "suggested_action": "Ask the customer to clarify their specific refund concern and provide the relevant order number, purchase date, and product or service details.",
  "reasoning": "The content is safe and contains no harmful or offensive material. It focuses on refunds, cancellations, and a money-back guarantee, so Billing is the most appropriate category. Its reassuring, helpful language indicates a Positive sentiment, although it does not describe a specific customer problem."
}


## 6. Batch Processing (PoC Execution)
Iterating over the 50 sampled tickets and storing the structured responses.

In [46]:
results_list = []

for index,row in tqdm(df_sample.iterrows(), total= df_sample.shape[0], desc = "Processing Tickets"):
    ticket_text = row[CONFIG["text_col"]]

    try:
        res = client.chat.completions.create(
            model = CONFIG["model_name"],
            messages = [ 
                {"role" : "system", "content" : SYSTEM_PROMPT},
                {"role" : "user", "content" : create_user_prompt(ticket_text)},   
            ],
            temperature=0.0,
            response_format = {"type": "json_object"}
        )

        raw_json = res.choices[0].message.content
        parsed = TicketProcessingResult.model_validate_json(raw_json)

        # Merge original data with LLM output
        combined_row = row.to_dict()
        combined_row.update(parsed.model_dump())
        results_list.append(combined_row) 

    except Exception as e:
        print(f"Error processing row {index} : {e}")

        # Add empty data for failed row to maintain alignment
        combined_row = row.to_dict()
        combined_row.update({k:None for k in TicketProcessingResult.model_fields.keys()})

        results_list.append(combined_row)

df_results = pd.DataFrame(results_list)
print(f"Complete processing.\nResults shape: {df_results.shape}")

Processing Tickets:   0%|          | 0/50 [00:00<?, ?it/s]

Processing Tickets: 100%|██████████| 50/50 [02:53<00:00,  3.47s/it]

Complete processing.
Results shape: (50, 12)


## 7. Export Results
Saving the evaluated dataset for manual review or LLM-as-a-Judge evaluation.

In [47]:
# Ensure directory exists
os.makedirs(os.path.dirname(CONFIG["output_results_file"]),exist_ok = True)

In [48]:
# Export to csv
df_results.to_csv(CONFIG["output_results_file"],index = False)
print(f"Final results successfully saved to: {CONFIG["output_results_file"]}")

df_results.head(2)

Final results successfully saved to: data/poc_results_50.csv


,flags,instruction,category,intent,response,text_length,noise_score,is_safe,sentiment,summary,suggested_action,reasoning
0,BCL,"I want to check your money back guarantee, I n...",Billing,check_refund_policy,Definitely! I completely understand your desir...,2424,0.035066,True,Positive,The ticket discusses the conditions of a money...,Ask the customer to clarify their specific ref...,The content is safe and contains no harmful or...
1,BL,help me seeing what hours I can contact custom...,General Inquiry,contact_customer_service,Grateful for your contact! I get the sense tha...,457,0.024070,True,Positive,The ticket concerns the hours during which cus...,Replace the {{Customer Support Hours}} placeho...,The content is safe and contains no harmful or...
